Install Dependencies & Import Libraries

In [24]:
!pip -q install albumentations opencv-python-headless tqdm scikit-learn

In [25]:
import os
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import albumentations as A

from sklearn.model_selection import train_test_split

In [26]:
ROLL_NUMBER_LAST_3 = 19
SEED = 19

print("Roll-number seed:", SEED)

Roll-number seed: 19


In [27]:
def seed_everything(seed=19):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)



    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print("Seed configured successfully.")

Seed configured successfully.


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available.")

Device: cuda
GPU: Tesla T4


Upload Dataset ZIP File

In [29]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

KeyboardInterrupt: 

Extract Dataset from ZIP

In [30]:
import zipfile

zip_files = list(Path(".").glob("*.zip"))

if not zip_files:
    raise FileNotFoundError(
        "No ZIP file found. Please upload the real NEU-Seg dataset ZIP."
    )

zip_path = zip_files[0]

print("Using ZIP:", zip_path)

extract_dir = Path("/content/neu_seg_data")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extraction complete.")

FileNotFoundError: No ZIP file found. Please upload the real NEU-Seg dataset ZIP.

Inspect dataset

In [ ]:
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(str(extract_dir), "").count(os.sep)

    if level <= 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

        for f in files[:10]:
            print(f"{indent}  {f}")

        if len(files) > 10:
            print(f"{indent}  ... {len(files)-10} more files")

Explore Sample XML Annotation

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

annotation_dir = Path(
    "/content/neu_seg_data/NEU-DET/train/annotations"
)

xml_files = list(annotation_dir.glob("*.xml"))

print("Number of XML files:", len(xml_files))

if not xml_files:
    raise FileNotFoundError("No XML annotations found.")

sample_xml = xml_files[0]

print("\nSample XML:")
print("=" * 80)

print(
    sample_xml.read_text(
        encoding="utf-8"
    )[:5000]
)

In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import cv2
from pathlib import Path


def xml_to_binary_mask(xml_path, image_size=(200, 200)):
    """
    Convert Pascal VOC bounding-box annotations
    into a binary pseudo-mask.

    0 = background
    1 = inside any annotated bounding box
    """

    width, height = image_size

    mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for obj in root.findall("object"):

        bbox = obj.find("bndbox")

        if bbox is None:
            continue

        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        # Keep coordinates inside image
        xmin = max(0, min(xmin, width - 1))
        xmax = max(0, min(xmax, width - 1))

        ymin = max(0, min(ymin, height - 1))
        ymax = max(0, min(ymax, height - 1))

        # Fix reversed coordinates if necessary
        if xmin > xmax:
            xmin, xmax = xmax, xmin

        if ymin > ymax:
            ymin, ymax = ymax, ymin

        # Fill bounding-box region
        mask[ymin:ymax + 1, xmin:xmax + 1] = 1

    return mask

In [ ]:
sample_xml = (
    "/content/neu_seg_data/"
    "NEU-DET/train/annotations/"
    "scratches_16.xml"
)

sample_mask = xml_to_binary_mask(
    sample_xml,
    image_size=(200, 200)
)

print("Mask shape:", sample_mask.shape)
print("Unique values:", np.unique(sample_mask))
print(
    "Defect pixels:",
    int(sample_mask.sum())
)

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. Dataset directories
# ============================================================

BASE_DIR = Path("/content/neu_seg_data/NEU-DET")

TRAIN_IMAGE_DIR = BASE_DIR / "train" / "images"
TRAIN_ANNOTATION_DIR = BASE_DIR / "train" / "annotations"


# ============================================================
# 2. Select an XML annotation
# ============================================================

sample_xml_path = (
    TRAIN_ANNOTATION_DIR / "scratches_16.xml"
)

if not sample_xml_path.exists():
    raise FileNotFoundError(
        f"XML not found: {sample_xml_path}"
    )


# ============================================================
# 3. Read filename directly from XML
# ============================================================

tree = ET.parse(sample_xml_path)
root = tree.getroot()

filename_element = root.find("filename")

if filename_element is None:
    raise ValueError(
        "The XML file does not contain a <filename> element."
    )

xml_filename = filename_element.text.strip()

print("Filename written inside XML:")
print(xml_filename)


# ============================================================
# 4. Search for the actual image
# ============================================================

# First try exact filename
possible_image = TRAIN_IMAGE_DIR / xml_filename

if possible_image.exists():

    image_path = possible_image

else:

    # If extension/name differs, search by filename stem
    stem = Path(xml_filename).stem

    candidates = [
        p for p in TRAIN_IMAGE_DIR.rglob("*")
        if p.is_file()
        and p.stem.lower() == stem.lower()
    ]

    if len(candidates) == 0:

        raise FileNotFoundError(
            f"No image matching '{xml_filename}' "
            f"was found inside {TRAIN_IMAGE_DIR}"
        )

    image_path = candidates[0]


print("\nActual image found:")
print(image_path)


# ============================================================
# 5. Read image
# ============================================================

image = cv2.imread(
    str(image_path),
    cv2.IMREAD_GRAYSCALE
)

if image is None:
    raise ValueError(
        f"OpenCV could not read: {image_path}"
    )

image = np.asarray(
    image,
    dtype=np.uint8
)

print("\nImage information:")
print("Shape :", image.shape)
print("Dtype :", image.dtype)
print("Min   :", image.min())
print("Max   :", image.max())


# ============================================================
# 6. Convert XML bounding boxes to pseudo-mask
# ============================================================

height, width = image.shape

sample_mask = np.zeros(
    (height, width),
    dtype=np.uint8
)

objects = root.findall("object")

print("\nObjects found in XML:", len(objects))

for i, obj in enumerate(objects, start=1):

    name_element = obj.find("name")
    bbox = obj.find("bndbox")

    if bbox is None:
        print(
            f"Object {i}: no bounding box found"
        )
        continue

    defect_name = (
        name_element.text.strip()
        if name_element is not None
        else "unknown"
    )

    xmin = int(
        bbox.find("xmin").text
    )

    ymin = int(
        bbox.find("ymin").text
    )

    xmax = int(
        bbox.find("xmax").text
    )

    ymax = int(
        bbox.find("ymax").text
    )

    # Keep coordinates inside image
    xmin = max(
        0,
        min(xmin, width - 1)
    )

    xmax = max(
        0,
        min(xmax, width - 1)
    )

    ymin = max(
        0,
        min(ymin, height - 1)
    )

    ymax = max(
        0,
        min(ymax, height - 1)
    )

    # Handle reversed coordinates
    if xmin > xmax:
        xmin, xmax = xmax, xmin

    if ymin > ymax:
        ymin, ymax = ymax, ymin

    # Fill the annotated bounding-box region
    sample_mask[
        ymin:ymax + 1,
        xmin:xmax + 1
    ] = 1

    print(
        f"Object {i}: {defect_name} | "
        f"Box = ({xmin}, {ymin}, {xmax}, {ymax})"
    )


print("\nMask information:")
print("Shape:", sample_mask.shape)
print("Dtype:", sample_mask.dtype)
print(
    "Unique values:",
    np.unique(sample_mask)
)


# ============================================================
# 7. Create visualization overlay
# ============================================================

overlay = cv2.cvtColor(
    image,
    cv2.COLOR_GRAY2RGB
)

overlay[
    sample_mask == 1
] = [255, 0, 0]


# ============================================================
# 8. Display
# ============================================================

plt.figure(
    figsize=(15, 5)
)

plt.subplot(1, 3, 1)

plt.imshow(
    image,
    cmap="gray"
)

plt.title(
    "Original NEU-DET Image"
)

plt.axis("off")


plt.subplot(1, 3, 2)

plt.imshow(
    sample_mask,
    cmap="gray"
)

plt.title(
    "Bounding-Box Pseudo-Mask"
)

plt.axis("off")


plt.subplot(1, 3, 3)

plt.imshow(
    overlay
)

plt.title(
    "Image + Annotation"
)

plt.axis("off")


plt.tight_layout()
plt.show()

Define PyTorch Dataset Class

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


BASE_DIR = Path("/content/neu_seg_data/NEU-DET")

TRAIN_IMG_DIR = BASE_DIR / "train" / "images"
TRAIN_XML_DIR = BASE_DIR / "train" / "annotations"

VAL_IMG_DIR = BASE_DIR / "validation" / "images"
VAL_XML_DIR = BASE_DIR / "validation" / "annotations"


class NEUDataset(Dataset):

    def __init__(self, image_dir, annotation_dir):
        self.image_dir = Path(image_dir)
        self.annotation_dir = Path(annotation_dir)

        self.samples = []

        for xml_path in sorted(self.annotation_dir.glob("*.xml")):

            tree = ET.parse(xml_path)
            root = tree.getroot()

            filename_node = root.find("filename")

            if filename_node is None:
                continue

            filename = filename_node.text.strip()
            stem = Path(filename).stem

            # Find matching image using filename stem
            candidates = [
                p for p in self.image_dir.rglob("*")
                if p.is_file()
                and p.stem.lower() == stem.lower()
            ]

            if len(candidates) == 0:
                continue

            self.samples.append(
                (candidates[0], xml_path)
            )

        print(
            f"Loaded {len(self.samples)} image/XML pairs"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_path, xml_path = self.samples[idx]

        # Read real image
        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            raise ValueError(
                f"Could not read image: {image_path}"
            )

        height, width = image.shape

        # Normalize image
        image = image.astype(
            np.float32
        ) / 255.0

        # Convert XML boxes to pseudo-mask
        mask = np.zeros(
            (height, width),
            dtype=np.float32
        )

        tree = ET.parse(xml_path)
        root = tree.getroot()

        for obj in root.findall("object"):

            bbox = obj.find("bndbox")

            if bbox is None:
                continue

            xmin = int(bbox.find("xmin").text)
            ymin = int(bbox.find("ymin").text)
            xmax = int(bbox.find("xmax").text)
            ymax = int(bbox.find("ymax").text)

            xmin = max(0, min(xmin, width - 1))
            xmax = max(0, min(xmax, width - 1))
            ymin = max(0, min(ymin, height - 1))
            ymax = max(0, min(ymax, height - 1))

            if xmin > xmax:
                xmin, xmax = xmax, xmin

            if ymin > ymax:
                ymin, ymax = ymax, ymin

            mask[
                ymin:ymax + 1,
                xmin:xmax + 1
            ] = 1.0

        # Convert to PyTorch format
        image = torch.tensor(
            image,
            dtype=torch.float32
        ).unsqueeze(0)

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).unsqueeze(0)

        return image, mask

Train and Validate the dataset


In [ ]:
train_dataset = NEUDataset(
    TRAIN_IMG_DIR,
    TRAIN_XML_DIR
)

val_dataset = NEUDataset(
    VAL_IMG_DIR,
    VAL_XML_DIR
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

DataLoaders (Train & Validation)

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Sanity Check: Inspect One Batch

In [ ]:
images, masks = next(iter(train_loader))

print("Images shape:", images.shape)
print("Masks shape :", masks.shape)

print("Image min:", images.min().item())
print("Image max:", images.max().item())

print("Mask unique values:", torch.unique(masks))

Initialize Model and Move to Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

model = ResUNet().to(device)

print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

Loss function Binary Cross Entropy

In [ ]:
class DiceLoss(nn.Module):

    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(
            probs.size(0),
            -1
        )

        targets = targets.view(
            targets.size(0),
            -1
        )

        intersection = (
            probs * targets
        ).sum(dim=1)

        dice = (
            2 * intersection
            + self.smooth
        ) / (
            probs.sum(dim=1)
            + targets.sum(dim=1)
            + self.smooth
        )

        return 1 - dice.mean()


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()


def combined_loss(logits, targets):

    bce = bce_loss(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    return bce + dice

Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Optimizer: Adam")
print("Learning rate:", 1e-3)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ResUNet().to(device)

print("Device:", device)
print("Model recreated successfully.")

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    output = model(images)

print("Input :", images.shape)
print("Target:", masks.shape)
print("Output:", output.shape)

In [ ]:
loss = combined_loss(
    output,
    masks
)

print("Test loss:", loss.item())

Reinitialize Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Adam + LR 1e-3 ready")

Train ResUNet

In [ ]:
import time
import torch

NUM_EPOCHS = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    model.train()
    train_total = 0.0

    start = time.time()

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        train_total += loss.item() * images.size(0)

    train_loss = train_total / len(train_dataset)

    # Validation
    model.eval()
    val_total = 0.0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)

            loss = combined_loss(logits, masks)

            val_total += loss.item() * images.size(0)

    val_loss = val_total / len(val_dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "/content/best_resunet_neu.pth"
        )

        marker = " BEST"

    else:
        marker = ""

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f} | "
        f"{time.time()-start:.1f}s"
        f"{marker}"
    )

Load Best Saved Model Weights

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/best_resunet_neu.pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

Define Evaluation Metrics (IoU, Dice, Precision, Recall)

In [ ]:
import numpy as np
import torch

def get_metrics(logits, targets, threshold=0.5, eps=1e-7):

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.flatten(1)
    targets = targets.flatten(1)

    tp = (preds * targets).sum(1)
    fp = (preds * (1 - targets)).sum(1)
    fn = ((1 - preds) * targets).sum(1)

    iou = (tp + eps) / (tp + fp + fn + eps)

    dice = (2 * tp + eps) / (
        2 * tp + fp + fn + eps
    )

    precision = (tp + eps) / (
        tp + fp + eps
    )

    recall = (tp + eps) / (
        tp + fn + eps
    )

    return (
        iou.mean().item(),
        dice.mean().item(),
        precision.mean().item(),
        recall.mean().item()
    )


results = []

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)

        results.append(
            get_metrics(logits, masks)
        )

results = np.array(results)

print("\n===== REAL VALIDATION RESULTS =====")

print(f"IoU       : {results[:,0].mean():.4f}")
print(f"Dice      : {results[:,1].mean():.4f}")
print(f"Precision : {results[:,2].mean():.4f}")
print(f"Recall    : {results[:,3].mean():.4f}")

Evaluate ResUNet on Validation Set

In [ ]:
import matplotlib.pyplot as plt

images, masks = next(iter(val_loader))

with torch.no_grad():

    logits = model(
        images.to(device)
    )

preds = (
    torch.sigmoid(logits) >= 0.5
).float()

n = min(3, images.size(0))

plt.figure(figsize=(12, 4*n))

for i in range(n):

    plt.subplot(n, 3, i*3+1)
    plt.imshow(
        images[i,0].numpy(),
        cmap="gray"
    )
    plt.title("Real NEU-DET Image")
    plt.axis("off")

    plt.subplot(n, 3, i*3+2)
    plt.imshow(
        masks[i,0].numpy(),
        cmap="gray"
    )
    plt.title("XML Pseudo-Mask")
    plt.axis("off")

    plt.subplot(n, 3, i*3+3)
    plt.imshow(
        preds[i,0].cpu().numpy(),
        cmap="gray"
    )
    plt.title("ResUNet Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

Visualize Predictions vs Ground Truth (ResUNet)

In [ ]:
import time
import torch

model.eval()

# Warm-up
with torch.no_grad():
    for images, _ in list(val_loader)[:2]:
        images = images.to(device)
        _ = model(images)

        if device.type == "cuda":
            torch.cuda.synchronize()

# Timing
count = 0

if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()

with torch.no_grad():

    for images, _ in val_loader:

        images = images.to(device)
        _ = model(images)

        count += images.size(0)

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start

print(f"Images: {count}")
print(f"Time: {elapsed:.4f}s")
print(f"FPS: {count / elapsed:.2f}")

CNN Encoder-Decoder

In [ ]:
class StandardCNNEncoderDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32,
                kernel_size=2,
                stride=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 16,
                kernel_size=2,
                stride=2
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(
                16, 8,
                kernel_size=2,
                stride=2
            ),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(
                8, 1,
                kernel_size=1
            )
        )

    def forward(self, x):

        x = self.encoder(x)
        x = self.decoder(x)

        # Guarantee original resolution
        x = F.interpolate(
            x,
            size=(200, 200),
            mode="bilinear",
            align_corners=False
        )

        return x

In [ ]:
baseline_model = StandardCNNEncoderDecoder().to(device)

baseline_optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=1e-3
)

print("Standard CNN Encoder-Decoder ready.")

Autoencoder

In [ ]:
class ConvAutoencoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(
                1, 16,
                3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv2d(
                16, 32,
                3,
                stride=2,
                padding=1
            ),

            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(
                32, 16,
                3,
                stride=2,
                padding=1,
                output_padding=1
            ),

            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.ConvTranspose2d(
                16, 1,
                3,
                stride=2,
                padding=1,
                output_padding=1
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        return self.decoder(
            self.encoder(x)
        )

# Training of 3 Models
42 Epoch

In [ ]:
import torch
import torch.nn as nn
import time

# ==========================================
# MODELS
# ==========================================

baseline_model = baseline_model.to(device)

autoencoder = ConvAutoencoder().to(device)

model = model.to(device)   # ResUNet


# ==========================================
# OPTIMIZERS
# ==========================================

optimizer_baseline = torch.optim.Adam(
    baseline_model.parameters(),
    lr=1e-3
)

optimizer_AE = torch.optim.Adam(
    autoencoder.parameters(),
    lr=1e-3
)

optimizer_UNet = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


# ==========================================
# LOSSES
# ==========================================

mse_loss = nn.MSELoss()


# ==========================================
# QUICK TRAINING
# ==========================================

EPOCHS = 42

baseline_history = []
ae_history = []
resunet_history = []


for epoch in range(EPOCHS):

    baseline_model.train()
    autoencoder.train()
    model.train()

    baseline_total = 0
    ae_total = 0
    resunet_total = 0

    start = time.time()

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)


        # ----------------------------------
        # 1. STANDARD CNN ENCODER-DECODER
        # ----------------------------------

        optimizer_baseline.zero_grad()

        baseline_output = baseline_model(images)

        baseline_loss = combined_loss(
            baseline_output,
            masks
        )

        baseline_loss.backward()

        optimizer_baseline.step()

        baseline_total += baseline_loss.item()


        # ----------------------------------
        # 2. CONVOLUTIONAL AUTOENCODER
        # ----------------------------------

        optimizer_AE.zero_grad()

        reconstructed = autoencoder(images)

        ae_loss = mse_loss(
            reconstructed,
            images
        )

        ae_loss.backward()

        optimizer_AE.step()

        ae_total += ae_loss.item()


        # ----------------------------------
        # 3. RESUNET
        # ----------------------------------

        optimizer_UNet.zero_grad()

        resunet_output = model(images)

        resunet_loss = combined_loss(
            resunet_output,
            masks
        )

        resunet_loss.backward()

        optimizer_UNet.step()

        resunet_total += resunet_loss.item()


    # Average losses

    baseline_avg = (
        baseline_total / len(train_loader)
    )

    ae_avg = (
        ae_total / len(train_loader)
    )

    resunet_avg = (
        resunet_total / len(train_loader)
    )

    baseline_history.append(
        baseline_avg
    )

    ae_history.append(
        ae_avg
    )

    resunet_history.append(
        resunet_avg
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"CNN Loss: {baseline_avg:.4f} | "
        f"AE Loss: {ae_avg:.4f} | "
        f"ResUNet Loss: {resunet_avg:.4f} | "
        f"Time: {time.time()-start:.1f}s"
    )


print("\nTraining completed.")

Real validation comparison

In [ ]:
import numpy as np
import torch


def segmentation_metrics(
    logits,
    targets,
    threshold=0.5,
    eps=1e-7
):

    probs = torch.sigmoid(logits)

    preds = (
        probs >= threshold
    ).float()

    preds = preds.flatten(1)
    targets = targets.flatten(1)

    tp = (
        preds * targets
    ).sum(dim=1)

    fp = (
        preds * (1 - targets)
    ).sum(dim=1)

    fn = (
        (1 - preds) * targets
    ).sum(dim=1)

    iou = (
        tp + eps
    ) / (
        tp + fp + fn + eps
    )

    dice = (
        2 * tp + eps
    ) / (
        2 * tp + fp + fn + eps
    )

    precision = (
        tp + eps
    ) / (
        tp + fp + eps
    )

    recall = (
        tp + eps
    ) / (
        tp + fn + eps
    )

    return [
        iou.mean().item(),
        dice.mean().item(),
        precision.mean().item(),
        recall.mean().item()
    ]


# ==========================================
# Evaluation
# ==========================================

baseline_model.eval()
model.eval()

baseline_results = []
resunet_results = []

ae_validation_loss = []


with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)


        # CNN baseline

        baseline_logits = baseline_model(
            images
        )

        baseline_results.append(
            segmentation_metrics(
                baseline_logits,
                masks
            )
        )


        # ResUNet

        resunet_logits = model(
            images
        )

        resunet_results.append(
            segmentation_metrics(
                resunet_logits,
                masks
            )
        )


        # Autoencoder

        reconstructed = autoencoder(
            images
        )

        ae_loss = mse_loss(
            reconstructed,
            images
        )

        ae_validation_loss.append(
            ae_loss.item()
        )


baseline_results = np.array(
    baseline_results
)

resunet_results = np.array(
    resunet_results
)


print("\n========== REAL VALIDATION RESULTS ==========")

print("\nSTANDARD CNN ENCODER-DECODER")

print(
    f"IoU       : {baseline_results[:,0].mean():.4f}"
)

print(
    f"Dice      : {baseline_results[:,1].mean():.4f}"
)

print(
    f"Precision : {baseline_results[:,2].mean():.4f}"
)

print(
    f"Recall    : {baseline_results[:,3].mean():.4f}"
)


print("\nRESUNET")

print(
    f"IoU       : {resunet_results[:,0].mean():.4f}"
)

print(
    f"Dice      : {resunet_results[:,1].mean():.4f}"
)

print(
    f"Precision : {resunet_results[:,2].mean():.4f}"
)

print(
    f"Recall    : {resunet_results[:,3].mean():.4f}"
)


print("\nAUTOENCODER")

print(
    f"Reconstruction MSE: "
    f"{np.mean(ae_validation_loss):.4f}"
)

Visuals

In [ ]:
import matplotlib.pyplot as plt

images, masks = next(iter(val_loader))

images_gpu = images.to(device)

baseline_model.eval()
autoencoder.eval()
model.eval()

with torch.no_grad():

    baseline_pred = torch.sigmoid(
        baseline_model(images_gpu)
    )

    reconstruction = autoencoder(
        images_gpu
    )

    resunet_pred = torch.sigmoid(
        model(images_gpu)
    )


i = 0

plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(
    images[i, 0].numpy(),
    cmap="gray"
)
plt.title("Real NEU-DET Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(
    reconstruction[i, 0].cpu().numpy(),
    cmap="gray"
)
plt.title("Autoencoder Reconstruction")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(
    masks[i, 0].numpy(),
    cmap="gray"
)
plt.title("XML-derived Pseudo-Mask")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(
    baseline_pred[i, 0].cpu().numpy() >= 0.5,
    cmap="gray"
)
plt.title("CNN Segmentation")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(
    resunet_pred[i, 0].cpu().numpy() >= 0.5,
    cmap="gray"
)
plt.title("ResUNet Segmentation")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(
    images[i, 0].numpy(),
    cmap="gray"
)
plt.contour(
    masks[i, 0].numpy(),
    levels=[0.5]
)
plt.title("Ground Annotation Boundary")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("======================================")
print("FINAL REAL NEU-DET VALIDATION RESULTS")
print("======================================")

print("\nStandard CNN Encoder-Decoder")
print(f"IoU       : {baseline_results[:,0].mean():.4f}")
print(f"Dice      : {baseline_results[:,1].mean():.4f}")
print(f"Precision : {baseline_results[:,2].mean():.4f}")
print(f"Recall    : {baseline_results[:,3].mean():.4f}")

print("\nDeep ResUNet")
print(f"IoU       : {resunet_results[:,0].mean():.4f}")
print(f"Dice      : {resunet_results[:,1].mean():.4f}")
print(f"Precision : {resunet_results[:,2].mean():.4f}")
print(f"Recall    : {resunet_results[:,3].mean():.4f}")

print("\nConvolutional Autoencoder")
print(
    f"Reconstruction MSE : "
    f"{np.mean(ae_validation_loss):.4f}"
)

In [31]:
import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# 1. Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Complete Dataset Class Definition
class NEUDataset(Dataset):
    def __init__(self, image_dir, annotation_dir):
        self.image_dir = Path(image_dir)
        self.annotation_dir = Path(annotation_dir)
        self.samples = []

        if not self.annotation_dir.exists() or not self.image_dir.exists():
            print(f"Warning: Paths do not exist - {self.image_dir}")
            return

        for xml_path in sorted(self.annotation_dir.glob("*.xml")):
            tree = ET.parse(xml_path)
            root = tree.getroot()
            filename_node = root.find("filename")
            if filename_node is None: continue

            stem = Path(filename_node.text.strip()).stem
            candidates = [p for p in self.image_dir.rglob("*") if p.is_file() and p.stem.lower() == stem.lower()]
            if candidates:
                self.samples.append((candidates[0], xml_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, xml_path = self.samples[idx]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        h, w = image.shape
        image = image.astype(np.float32) / 255.0

        mask = np.zeros((h, w), dtype=np.float32)
        root = ET.parse(xml_path).getroot()
        for obj in root.findall("object"):
            bbox = obj.find("bndbox")
            if bbox is not None:
                xmin = max(0, min(int(bbox.find("xmin").text), w - 1))
                xmax = max(0, min(int(bbox.find("xmax").text), w - 1))
                ymin = max(0, min(int(bbox.find("ymin").text), h - 1))
                ymax = max(0, min(int(bbox.find("ymax").text), h - 1))
                if xmin > xmax: xmin, xmax = xmax, xmin
                if ymin > ymax: ymin, ymax = ymax, ymin
                mask[ymin:ymax + 1, xmin:xmax + 1] = 1.0

        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return image_tensor, mask_tensor

# 3. Safe Path Setup (Using Train directory to ensure data is found)
BASE_DIR = Path("/content/neu_seg_data/NEU-DET")
IMG_DIR = BASE_DIR / "train" / "images"
XML_DIR = BASE_DIR / "train" / "annotations"

try:
    # 4. Load Data & Models
    dataset = NEUDataset(IMG_DIR, XML_DIR)
    if len(dataset) == 0:
        raise ValueError("Dataset is totally empty! Zip file sahi se extract nahi hui ya paths ghalat hain.")

    # Sirf ek batch lenge visualization ke liye
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    images, masks = next(iter(loader))
    images_gpu = images.to(device)

    # Models ko memory mein check karte hue evaluation mode par lagana
    autoencoder.eval()
    model.eval() # ResUNet

    with torch.no_grad():
        reconstruction = autoencoder(images_gpu)
        resunet_pred = torch.sigmoid(model(images_gpu))

    # 5. Perfect Matplotlib Plotting for Output Screenshot
    i = 0
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(images[i, 0].numpy(), cmap="gray")
    plt.title("Real NEU-DET Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(reconstruction[i, 0].cpu().numpy(), cmap="gray")
    plt.title("Autoencoder Reconstruction")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(resunet_pred[i, 0].cpu().numpy() >= 0.5, cmap="gray")
    plt.title("ResUNet Segmentation Prediction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

except NameError as ne:
    print(f"Error: {ne}")
    print("Aapne model train nahi kiya! Colab session restart ho gaya hoga. Pehle autoencoder aur ResUNet (model) ko train wala cell run karein.")
except Exception as e:
    print(f"Error: {e}")

Error: Dataset is totally empty! Zip file sahi se extract nahi hui ya paths ghalat hain.
